# LeetCode Problem Scraper — with Descriptions

Scrapes **all problem names + full descriptions** from https://leetcode.com/problemset/  
using the LeetCode public GraphQL API.

- Free problems → full HTML description fetched and converted to plain text  
- Premium (paid-only) problems → description left blank  

> No login required. No Selenium. Just the public GraphQL endpoint.

In [1]:
# Install required libraries
!pip install requests pandas tqdm html2text --quiet

In [2]:
import requests
import pandas as pd
from tqdm import tqdm
import time
import json
import html2text

# html2text converter — strips HTML tags, keeps readable markdown-style text
h2t = html2text.HTML2Text()
h2t.ignore_links  = True
h2t.ignore_images = True
h2t.body_width    = 0   # no line-wrapping

In [3]:
# ── Configuration ──────────────────────────────────────────────────────────────
GRAPHQL_URL       = "https://leetcode.com/graphql"
PAGE_SIZE         = 100   # problems per list-page request
LIST_DELAY_SEC    = 0.5   # delay between list-page requests
DESC_DELAY_SEC    = 0.8   # delay between per-problem description requests
DESC_BATCH_SIZE   = 50    # checkpoint: save CSV every N descriptions fetched

HEADERS = {
    "Content-Type": "application/json",
    "User-Agent"  : "Mozilla/5.0 (compatible; LeetCode-Scraper/1.0)",
    "Referer"     : "https://leetcode.com/problemset/",
}

In [4]:
# ── Query 1: Problem list (paginated) ─────────────────────────────────────────
LIST_QUERY = """
query problemsetQuestionList($categorySlug: String, $limit: Int, $skip: Int, $filters: QuestionListFilterInput) {
  problemsetQuestionList: questionList(
    categorySlug: $categorySlug
    limit: $limit
    skip: $skip
    filters: $filters
  ) {
    total: totalNum
    questions: data {
      frontendQuestionId: questionFrontendId
      title
      titleSlug
      difficulty
      isPaidOnly
      topicTags { name }
    }
  }
}
"""

def fetch_problem_list_page(skip: int, limit: int = PAGE_SIZE) -> dict:
    payload = {
        "operationName": "problemsetQuestionList",
        "query"        : LIST_QUERY,
        "variables"    : {"categorySlug": "", "skip": skip, "limit": limit, "filters": {}}
    }
    r = requests.post(GRAPHQL_URL, json=payload, headers=HEADERS, timeout=15)
    r.raise_for_status()
    return r.json()

In [5]:
# ── Query 2: Single problem description ───────────────────────────────────────
DESC_QUERY = """
query questionContent($titleSlug: String!) {
  question(titleSlug: $titleSlug) {
    content
    mysqlSchemas
  }
}
"""

def fetch_description(slug: str) -> str:
    """Return plain-text description for a free problem, or '' on failure/premium."""
    payload = {
        "operationName": "questionContent",
        "query"        : DESC_QUERY,
        "variables"    : {"titleSlug": slug}
    }
    try:
        r = requests.post(GRAPHQL_URL, json=payload, headers=HEADERS, timeout=15)
        r.raise_for_status()
        data    = r.json()
        content = data.get("data", {}).get("question", {}).get("content") or ""
        if not content:
            return ""   # premium — content is null
        return h2t.handle(content).strip()
    except Exception:
        return ""

In [6]:
# ── Step 1: Fetch the full problem list ───────────────────────────────────────
first_page = fetch_problem_list_page(skip=0)
total      = first_page["data"]["problemsetQuestionList"]["total"]
print(f"Total problems on LeetCode: {total}")

all_problems = list(first_page["data"]["problemsetQuestionList"]["questions"])

for skip in tqdm(range(PAGE_SIZE, total, PAGE_SIZE), desc="Fetching problem list"):
    try:
        data = fetch_problem_list_page(skip=skip)
        all_problems.extend(data["data"]["problemsetQuestionList"]["questions"])
        time.sleep(LIST_DELAY_SEC)
    except requests.RequestException as e:
        print(f"\n⚠️  Error at skip={skip}: {e}. Retrying...")
        time.sleep(3)
        try:
            data = fetch_problem_list_page(skip=skip)
            all_problems.extend(data["data"]["problemsetQuestionList"]["questions"])
        except Exception as e2:
            print(f"   Failed again: {e2}")

print(f"\n✅ Collected metadata for {len(all_problems)} problems")

Total problems on LeetCode: 3865


Fetching problem list: 100%|██████████| 38/38 [00:43<00:00,  1.13s/it]


✅ Collected metadata for 3865 problems


In [7]:
# ── Step 2: Build base DataFrame ──────────────────────────────────────────────
rows = []
for q in all_problems:
    rows.append({
        "id"         : q.get("frontendQuestionId", ""),
        "title"      : q.get("title", ""),
        "slug"       : q.get("titleSlug", ""),
        "difficulty" : q.get("difficulty", ""),
        "is_paid"    : q.get("isPaidOnly", False),
        "tags"       : ", ".join(t["name"] for t in q.get("topicTags", [])),
        "url"        : f"https://leetcode.com/problems/{q.get('titleSlug', '')}/description/",
        "description": ""   # filled in next step
    })

df = pd.DataFrame(rows)
df["id"] = pd.to_numeric(df["id"], errors="coerce")
df = df.sort_values("id").reset_index(drop=True)

free_count = (~df["is_paid"]).sum()
paid_count = df["is_paid"].sum()
print(f"Free: {free_count}  |  Premium (skipped): {paid_count}")
df.head(5)

Free: 3107  |  Premium (skipped): 758


,id,title,slug,difficulty,is_paid,tags,url,description
0,1,Two Sum,two-sum,Easy,False,"Array, Hash Table",https://leetcode.com/problems/two-sum/descript...,
1,2,Add Two Numbers,add-two-numbers,Medium,False,"Linked List, Math, Recursion",https://leetcode.com/problems/add-two-numbers/...,
2,3,Longest Substring Without Repeating Characters,longest-substring-without-repeating-characters,Medium,False,"Hash Table, String, Sliding Window",https://leetcode.com/problems/longest-substrin...,
3,4,Median of Two Sorted Arrays,median-of-two-sorted-arrays,Hard,False,"Array, Binary Search, Divide and Conquer",https://leetcode.com/problems/median-of-two-so...,
4,5,Longest Palindromic Substring,longest-palindromic-substring,Medium,False,"Two Pointers, String, Dynamic Programming",https://leetcode.com/problems/longest-palindro...,


In [8]:
# ── Step 3: Fetch descriptions for free problems ──────────────────────────────
# Resume support: skip rows that already have a description
to_fetch = df[(~df["is_paid"]) & (df["description"] == "")].index.tolist()
print(f"Fetching descriptions for {len(to_fetch)} free problems…\n")

CHECKPOINT_FILE = "leetcode_problems_partial.csv"

for i, idx in enumerate(tqdm(to_fetch, desc="Fetching descriptions")):
    slug = df.at[idx, "slug"]
    desc = fetch_description(slug)
    df.at[idx, "description"] = desc
    time.sleep(DESC_DELAY_SEC)

    # Checkpoint every DESC_BATCH_SIZE rows
    if (i + 1) % DESC_BATCH_SIZE == 0:
        df.to_csv(CHECKPOINT_FILE, index=False)
        tqdm.write(f"  💾 Checkpoint saved ({i+1}/{len(to_fetch)})")

print(f"\n✅ Done. Descriptions fetched: {(df['description'] != '').sum()}")

Fetching descriptions for 3107 free problems…



Fetching descriptions:   2%|▏         | 50/3107 [01:05<1:04:40,  1.27s/it]

  💾 Checkpoint saved (50/3107)


Fetching descriptions:   3%|▎         | 100/3107 [02:09<1:02:37,  1.25s/it]

  💾 Checkpoint saved (100/3107)


...

  💾 Checkpoint saved (3050/3107)


Fetching descriptions: 100%|█████████▉| 3100/3107 [1:07:12<00:09,  1.40s/it]

  💾 Checkpoint saved (3100/3107)


Fetching descriptions: 100%|██████████| 3107/3107 [1:07:21<00:00,  1.30s/it]


✅ Done. Descriptions fetched: 3105


In [9]:
# ── Step 4: Preview a sample description ──────────────────────────────────────
sample = df[df["description"] != ""].iloc[3]
print(f"[{int(sample.id)}] {sample.title}  ({sample.difficulty})")
print("-" * 60)
print(sample.description[:800], "..." if len(sample.description) > 800 else "")

[4] Median of Two Sorted Arrays  (Hard)
------------------------------------------------------------
Given two sorted arrays `nums1` and `nums2` of size `m` and `n` respectively, return **the median** of the two sorted arrays.

The overall run time complexity should be `O(log (m+n))`.

 

**Example 1:**
    
    
    **Input:** nums1 = [1,3], nums2 = [2]
    **Output:** 2.00000
    **Explanation:** merged array = [1,2,3] and median is 2.
    

**Example 2:**
    
    
    **Input:** nums1 = [1,2], nums2 = [3,4]
    **Output:** 2.50000
    **Explanation:** merged array = [1,2,3,4] and median is (2 + 3) / 2 = 2.5.
    

 

**Constraints:**

  * `nums1.length == m`
  * `nums2.length == n`
  * `0 <= m <= 1000`
  * `0 <= n <= 1000`
  * `1 <= m + n <= 2000`
  * `-106 <= nums1[i], nums2[i] <= 106` 


In [10]:
# ── Step 5: Quick stats ────────────────────────────────────────────────────────
print("Difficulty breakdown:")
print(df["difficulty"].value_counts())

print("\nPaid-only vs free:")
print(df["is_paid"].value_counts().rename({True: "Paid", False: "Free"}))

print("\nDescriptions collected vs blank:")
print((df["description"] != "").value_counts().rename({True: "Has description", False: "Blank"}))

Difficulty breakdown:
difficulty
Medium    2022
Easy       930
Hard       913
Name: count, dtype: int64

Paid-only vs free:
is_paid
Free    3107
Paid     758
Name: count, dtype: int64

Descriptions collected vs blank:
description
Has description    3105
Blank               760
Name: count, dtype: int64


In [12]:
# ── Step 6: Save final CSV ─────────────────────────────────────────────────────
OUTPUT_FILE = "leetcode_problems_with_descriptions.csv"
df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {len(df)} rows → '{OUTPUT_FILE}'")

Saved 3865 rows → 'leetcode_problems_with_descriptions.csv'


In [13]:
# ── Step 7: (Optional) Full-text search across descriptions ───────────────────
# Example: find problems whose description mentions 'BFS'
keyword = "BFS"
mask = df["description"].str.contains(keyword, case=False, na=False)
df[mask][["id", "title", "difficulty", "tags"]].head(10)

,id,title,difficulty,tags
608,609,Find Duplicate File in System,Medium,"Array, Hash Table, String"


In [ ]:
# ── Step 8: (Optional) Export to JSON ─────────────────────────────────────────
JSON_FILE = "leetcode_problems_with_descriptions.json"
df.to_json(JSON_FILE, orient="records", indent=2, force_ascii=False)
print(f"Also saved as JSON → '{JSON_FILE}'")